# 🎲 Análisis Exploratorio de Datos (EDA) — FIDE Chess Dataset

**Evaluación 1 — AD 1.1: Ingestión y Exploración**

Este notebook **no entrena modelos**. Se limita a:
1. Cargar los datos crudos e intermedios desde el catálogo Kedro
2. Generar 3 gráficos clave: distribución de edades, distribución de ratings y jugadores por título
3. Análisis complementario de calidad y correlaciones

In [ ]:
# ============================================================
# Celda 1: Inicializar sesión de Kedro
# ============================================================
%load_ext kedro.ipython

In [ ]:
# ============================================================
# Celda 2: Imports y configuración de visualización
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo profesional para las gráficas
sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
print('✅ Librerías cargadas correctamente')

In [ ]:
# ============================================================
# Celda 3: Carga de datos desde el catálogo Kedro
# ============================================================
# Datos crudos (CSV originales)
players = catalog.load('fide_players')
ratings_2019 = catalog.load('fide_ratings_2019')
ratings_2020 = catalog.load('fide_ratings_2020')
ratings_2021 = catalog.load('fide_ratings_2021')

print('📋 Datasets cargados desde el catálogo Kedro:')
print(f'   Players:      {players.shape[0]:>10,} filas × {players.shape[1]} columnas')
print(f'   Ratings 2019: {ratings_2019.shape[0]:>10,} filas × {ratings_2019.shape[1]} columnas')
print(f'   Ratings 2020: {ratings_2020.shape[0]:>10,} filas × {ratings_2020.shape[1]} columnas')
print(f'   Ratings 2021: {ratings_2021.shape[0]:>10,} filas × {ratings_2021.shape[1]} columnas')

## 1. Perfil Inicial de los Datos

In [ ]:
# ============================================================
# Celda 4: Resumen rápido del dataset de jugadores
# ============================================================
print('=== PLAYERS ===')
print(f'Dimensiones: {players.shape}')
print(f'\nColumnas y tipos:')
print(players.dtypes)
print(f'\nPrimeras 5 filas:')
display(players.head())
print(f'\nEstadísticas descriptivas:')
display(players.describe(include='all'))

---
## 2. Gráficos Clave del EDA

Los tres gráficos principales que resumen las características más relevantes del dataset.

In [ ]:
# ============================================================
# GRÁFICO 1: Distribución de Año de Nacimiento (yob)
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))

yob_data = players['yob'].dropna().astype(int)

# Histograma con KDE superpuesto
ax.hist(yob_data, bins=80, color='#3498db', edgecolor='white',
        alpha=0.75, density=False, label='Frecuencia')

# Líneas de referencia
mean_yob = yob_data.mean()
median_yob = yob_data.median()
ax.axvline(mean_yob, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Media: {mean_yob:.0f}')
ax.axvline(median_yob, color='#2ecc71', linestyle='-.', linewidth=2,
           label=f'Mediana: {median_yob:.0f}')

ax.set_title('Distribución del Año de Nacimiento de Jugadores FIDE', fontweight='bold')
ax.set_xlabel('Año de Nacimiento')
ax.set_ylabel('Cantidad de Jugadores')
ax.legend(fontsize=11, loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n📊 Resumen estadístico de yob:')
print(f'   Mínimo: {yob_data.min()} | Máximo: {yob_data.max()}')
print(f'   Media:  {mean_yob:.1f} | Mediana: {median_yob:.0f}')
print(f'   Desv. Est.: {yob_data.std():.1f}')

In [ ]:
# ============================================================
# GRÁFICO 2: Distribución de Ratings ELO (Standard, por año)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

colors = ['#e67e22', '#9b59b6', '#1abc9c']
datasets_ratings = [
    ('2019', ratings_2019),
    ('2020', ratings_2020),
    ('2021', ratings_2021),
]

for ax, (year, df), color in zip(axes, datasets_ratings, colors):
    elo = df['rating_standard'].dropna()

    # Histograma
    ax.hist(elo, bins=80, color=color, edgecolor='white', alpha=0.8)

    # Líneas de referencia
    ax.axvline(elo.mean(), color='#e74c3c', linestyle='--', linewidth=2,
               label=f'Media: {elo.mean():.0f}')
    ax.axvline(2000, color='#27ae60', linestyle=':', linewidth=2,
               label='Umbral Experto (2000)')

    ax.set_title(f'ELO Estándar — {year}', fontweight='bold')
    ax.set_xlabel('Rating Estándar')
    ax.set_ylabel('Cantidad')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Distribución de Ratings ELO Estándar por Año',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRÁFICO 3: Cantidad de Jugadores por Título FIDE
# ============================================================
fig, ax = plt.subplots(figsize=(14, 7))

# Conteo de títulos (excluyendo nulos / sin título)
title_counts = players['title'].dropna().value_counts()

# Paleta de colores gradiente
palette = sns.color_palette('viridis', n_colors=len(title_counts))

bars = ax.bar(title_counts.index, title_counts.values, color=palette,
              edgecolor='white', linewidth=0.8)

# Etiquetas de valor sobre cada barra
for bar, val in zip(bars, title_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Cantidad de Jugadores por Título FIDE', fontweight='bold', fontsize=15)
ax.set_xlabel('Título FIDE')
ax.set_ylabel('Cantidad de Jugadores')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

# Texto informativo
total_with_title = title_counts.sum()
total_players = len(players)
pct = total_with_title / total_players * 100
ax.annotate(
    f'Solo el {pct:.1f}% de los jugadores\ntiene un título FIDE',
    xy=(0.95, 0.95), xycoords='axes fraction',
    ha='right', va='top', fontsize=11,
    bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8)
)

plt.tight_layout()
plt.show()

print(f'\n📊 Resumen de títulos:')
print(f'   Jugadores con título: {total_with_title:,} ({pct:.1f}%)')
print(f'   Jugadores sin título: {total_players - total_with_title:,}')
print(f'\n   Desglose:')
for title, count in title_counts.items():
    print(f'   {title:>5s}: {count:>8,} jugadores')

---
## 3. Análisis de Calidad: Nulos y Duplicados

In [ ]:
# ============================================================
# Gráfico de nulos por dataset
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

datasets = {
    'Players': players,
    'Ratings 2019': ratings_2019,
    'Ratings 2020': ratings_2020,
    'Ratings 2021': ratings_2021,
}

for ax, (name, df) in zip(axes, datasets.items()):
    null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)
    null_pct.plot.barh(ax=ax, color='#e74c3c', alpha=0.8)
    ax.set_title(f'% Nulos — {name}', fontweight='bold')
    ax.set_xlabel('% Nulos')

plt.tight_layout()
plt.show()

# Resumen numérico
print('\n📋 Resumen de calidad:')
for name, df in datasets.items():
    n_dup = df.duplicated().sum()
    n_null = df.isnull().sum().sum()
    print(f'   {name:15s} | Filas: {len(df):>10,} | Nulos: {n_null:>10,} | Duplicados: {n_dup:>6,}')

---
## 4. Correlaciones entre Tipos de Rating

In [ ]:
# ============================================================
# Heatmap de correlación entre ratings (2021)
# ============================================================
numeric_ratings = ratings_2021[['rating_standard', 'rating_rapid', 'rating_blitz']].dropna()

fig, ax = plt.subplots(figsize=(8, 6))
corr = numeric_ratings.corr()
sns.heatmap(corr, annot=True, cmap='RdYlBu_r', vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax, fmt='.3f',
            annot_kws={'fontsize': 13, 'fontweight': 'bold'})
ax.set_title('Correlación entre Tipos de Rating (2021)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Conclusiones del EDA

**Hallazgos principales:**

1. **Volumen de datos:** El dataset contiene ~433K jugadores y ~12M registros de ratings.
2. **Nulos:** La columna `title` tiene un alto porcentaje de nulos (la mayoría de jugadores no tiene título FIDE). Las columnas `rating_rapid` y `rating_blitz` también presentan nulos significativos.
3. **Distribuciones:** El rating estándar sigue una distribución aproximadamente normal con media alrededor de 1500-1600.
4. **Género:** Existe un desbalance significativo, con predominancia masculina.
5. **Correlaciones:** Los tres tipos de rating (standard, rapid, blitz) están altamente correlacionados.
6. **Outliers:** Se detectan valores atípicos en los extremos de los ratings (muy bajos o muy altos).

Estos hallazgos justifican las transformaciones que se aplican en los pipelines de limpieza y transformación.